In [ ]:
import os
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
import json
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import gc

from utils.preprocessing.preprocessing import *

In [ ]:
cur_dir = Path(os.getcwd())
proj_dir = cur_dir.resolve().parent.parent
RAW_DATA_PATH = proj_dir / "data" / "raw" / "files"
OUTPUT_DATA_PATH = proj_dir / "data" / "preprocessed"

In [ ]:
EXCLUDE_FILES = ["chb12_27", "chb12_28", "chb12_29"]

SFREQ = 256
WINDOW_SEC = 5
OVERLAP = 0.5

## Loading data

In [ ]:
files = load_files(base_path=RAW_DATA_PATH, exclude_files=EXCLUDE_FILES)

In [ ]:
files.keys()

## Elaboration and cleaning of the data

The filterbank spans the frequency range 0.5-25 Hz since most seizure and nonseizure EEG activity falls within this range.

In [ ]:
for name, file in files.items():
    print(f"\n|--- Elaborating patient {name} ---|")
    
    # Extrapolate channels name
    ch_names = file.ch_names
    
    # Apply band-pass filter
    file.filter(0.5, 25, verbose=False)
    
    # Extrapolate filtered data and times
    data_filtered = file.get_data()
    
    del file
    
    # Generate scaler
    scaler = RobustScaler()

    # Cleaning of missing channels
    data_ch_filt = data_dict(data_filtered, ch_names, scaler)
    
    # Computing the windows and overlapping size
    win_size = int(WINDOW_SEC*SFREQ)
    step = int(win_size*(1-OVERLAP))

    # retrieve data from each channel
    chs = list(data_ch_filt.keys())
    signals = np.array([data_ch_filt[ch] for ch in chs])
    
    del data_ch_filt
    
    print(f"\nShape of the signal: {signals.shape}")
    
    windows = []

    # Sampling the windows
    for start in range(0, signals.shape[1]-win_size, step):     
        window = signals[:, start:start+win_size]
        windows.append(window)
        
    del signals

    windows = np.array(windows)
    
    print(f"\nShape of the windows: {windows.shape}")
    
    base_name = name.split('_')[0]
    
    if not os.path.exists(os.path.join(OUTPUT_DATA_PATH, base_name)):
        os.makedirs(os.path.join(OUTPUT_DATA_PATH, base_name))
    
    target_path = OUTPUT_DATA_PATH / f"{name.split('_')[0]}" / f'{name}.npz'
    
    # save the file name
    np.savez(target_path, windows=windows, channels=chs)
    
    del windows
    gc.collect()
    
    print(f"\n|--- Finish elaborating patient {name} ---|")
    
print(f"\nFinish to elaborate all the files")